In [ ]:
import os
import json
from pathlib import Path

# ==============================================================================
# ⚙️ CẤU HÌNH ĐÁNH GIÁ (CHỈ CẦN CHỌN TẠI ĐÂY)
# ==============================================================================
# Bạn chỉ cần thay đổi biến EVAL_LANGS ở dòng dưới:
#   EVAL_LANGS = ["vi"]          -> Chỉ đánh giá tập Tiếng Việt
#   EVAL_LANGS = ["en"]          -> Chỉ đánh giá tập Tiếng Anh
#   EVAL_LANGS = ["vi", "en"]    -> Tự động đánh giá cả 2 và in bảng so sánh tổng hợp
EVAL_LANGS = ["vi", "en"]

EXPERIMENT = "e3"

# Chọn model tương ứng với checkpoint bạn đã train bên Colab (2B hoặc 4B):
MODEL_ID = "unsloth/Qwen3.5-2B"  # Hoặc "unsloth/Qwen3.5-4B"

HF_NAME = "ThinhDao"
MODEL_NAME_ONLY = f"{MODEL_ID.rsplit('/', 1)[-1]}_{EXPERIMENT.upper()}"
REPO_HF_MODEL = f"{HF_NAME}/{MODEL_NAME_ONLY}"
ADAPTER_ID = REPO_HF_MODEL  # LoRA Adapter repo trên Hugging Face được Colab đẩy lên

RUN_NAME = f"{EXPERIMENT}_{MODEL_ID.rsplit('/', 1)[-1].lower()}"

# Tự động nhận diện đường dẫn DATA_ROOT trên Kaggle
candidate_paths = [
    Path("/kaggle/input/datasets/phcthnho/tool-calling-vi-experiments"),
    Path("/kaggle/input/tool-calling-vi-experiments"),
]
DATA_ROOT = next((p for p in candidate_paths if p.is_dir()), candidate_paths[0])

REVISION = "2026-09-02-full-dedup-seed42"
REVISION_DIR = DATA_ROOT / "benchmark_core" / REVISION

RUN_DIR = Path("/kaggle/working") / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Đồng bộ cấu hình sang biến môi trường để worker.py tự động nhận diện khi đổi EXPERIMENT
os.environ["EXPERIMENT"] = EXPERIMENT
os.environ["MODEL_ID"] = MODEL_ID
os.environ["HF_NAME"] = HF_NAME
os.environ["ADAPTER_ID"] = ADAPTER_ID
os.environ["RUN_NAME"] = RUN_NAME
os.environ["RUN_DIR"] = str(RUN_DIR)
os.environ["REVISION_DIR"] = str(REVISION_DIR)
CUSTOM_DIR = DATA_ROOT / "custom_vi"
os.environ["DATA_ROOT"] = str(DATA_ROOT)
os.environ["CUSTOM_DIR"] = str(CUSTOM_DIR)

print("="*60)
print(f"EXPERIMENT      : {EXPERIMENT}")
print(f"MODEL_ID        : {MODEL_ID}")
print(f"ADAPTER_ID      : {ADAPTER_ID}")
print(f"RUN_NAME        : {RUN_NAME}")
print(f"DATA_ROOT       : {DATA_ROOT}")
print(f"RUN_DIR         : {RUN_DIR}")
print("="*60)

def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as source:
        return [json.loads(line) for line in source if line.strip()]

In [ ]:
assert DATA_ROOT.is_dir(), DATA_ROOT
assert REVISION_DIR.is_dir(), REVISION_DIR

revision_manifest = json.loads(
    (REVISION_DIR / "manifest.json").read_text(encoding="utf-8")
)

assert revision_manifest["revision"] == REVISION

metadata = json.loads(
    (REVISION_DIR / "metadata.json").read_text(encoding="utf-8")
)

for lang in EVAL_LANGS:
    count = metadata["split_counts"][lang]["test"]
    assert count == 7712, f"Split count mismatch for {lang}: {count}"

    assert (REVISION_DIR / f"{lang}/test.jsonl").is_file(), \
        f"File {lang}/test.jsonl không tồn tại"

print(f"E3 preflight PASS cho: {EVAL_LANGS}")

In [ ]:
%pip install -q -U peft "transformers>=5.2.0" "accelerate>=1.0" "bitsandbytes>=0.43"

In [ ]:
import torch

assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator first"
for index in range(torch.cuda.device_count()):
    print(f"GPU {index}: {torch.cuda.get_device_name(index)}")

In [ ]:
SYSTEM_PROMPTS = {
    "en": "You are an AI assistant capable of using tools.",
    "vi": "Bạn là trợ lý AI có khả năng sử dụng công cụ.",
}

def format_tool(tool: dict) -> dict:
    return {
        "type": "function",
        "function": {
            "name": tool["name"],
            "description": tool.get("description", ""),
            "parameters": tool.get("parameters", {}),
        },
    }

def format_call(call: dict) -> dict:
    return {
        "type": "function",
        "function": {
            "name": call["name"],
            "arguments": call.get("arguments", {}),
        },
    }

def native_row(record: dict, language: str) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPTS[language]},
        {"role": "user", "content": record["query"]},
    ]
    calls = [format_call(call) for call in record["function_calls"]]
    if calls:
        messages.append({"role": "assistant", "content": "", "tool_calls": calls})
    else:
        fallback = "Hiện tại tôi chưa thể thực hiện yêu cầu này." if language == "vi" else "I cannot complete that request right now."
        messages.append({"role": "assistant", "content": record.get("assistant_content") or fallback})
    return {
        "id": record["id"],
        "messages": messages,
        "tools": [format_tool(tool) for tool in record["tools"]],
    }

from transformers import AutoTokenizer

tok_id = ADAPTER_ID if ADAPTER_ID else MODEL_ID
smoke_tokenizer = AutoTokenizer.from_pretrained(tok_id, trust_remote_code=True)
for lang in EVAL_LANGS:
    for record in read_jsonl(REVISION_DIR / f"{lang}/test.jsonl")[:3]:
        row = native_row(record, lang)
        rendered = smoke_tokenizer.apply_chat_template(
            row["messages"], tools=row["tools"], tokenize=False,
            add_generation_prompt=False, enable_thinking=False,
        )
        expected_calls = len(record["function_calls"])
        assistant_part = rendered.rsplit("<|im_start|>assistant", 1)[-1]
        assert assistant_part.count("<tool_call>") == expected_calls, record["id"]
print(f"native tokenizer smoke test PASS cho: {EVAL_LANGS}")

In [ ]:
%%writefile worker.py
import os
import sys
import json
import time
from pathlib import Path
import torch
from transformers import AutoModelForImageTextToText, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

gpu_id = int(sys.argv[1])
total_gpus = int(sys.argv[2])
split_lang = sys.argv[3] if len(sys.argv) > 3 else "vi"

# Tự động nhận diện cấu hình từ Cell 0 thông qua os.environ
EXPERIMENT = os.environ.get("EXPERIMENT", "e3")
MODEL_ID = os.environ.get("MODEL_ID", "unsloth/Qwen3.5-2B")
HF_NAME = os.environ.get("HF_NAME", "ThinhDao")
MODEL_NAME_ONLY = f"{MODEL_ID.rsplit('/', 1)[-1]}_{EXPERIMENT.upper()}"
ADAPTER_ID = os.environ.get("ADAPTER_ID", f"{HF_NAME}/{MODEL_NAME_ONLY}")

RUN_NAME = os.environ.get("RUN_NAME", f"{EXPERIMENT}_{MODEL_ID.rsplit('/', 1)[-1].lower()}")
RUN_DIR = Path(os.environ.get("RUN_DIR", f"/kaggle/working/{RUN_NAME}"))
RUN_DIR.mkdir(parents=True, exist_ok=True)

candidate_paths = [
    Path("/kaggle/input/datasets/phcthnho/tool-calling-vi-experiments"),
    Path("/kaggle/input/tool-calling-vi-experiments"),
]
default_data_root = next((p for p in candidate_paths if p.is_dir()), candidate_paths[0])
REVISION_DIR = Path(os.environ.get("REVISION_DIR", str(default_data_root / "benchmark_core/2026-09-02-full-dedup-seed42")))

PART_PATH = RUN_DIR / f"eval_predictions_{split_lang}_part_{gpu_id}.jsonl"

BATCH_SIZE = 16
MAX_NEW_TOKENS = 128

SYSTEM_PROMPTS = {
    "en": "You are an AI assistant capable of using tools.",
    "vi": "Bạn là trợ lý AI có khả năng sử dụng công cụ.",
}

def format_tool(tool: dict) -> dict:
    return {"type": "function", "function": {"name": tool["name"], "description": tool.get("description", ""), "parameters": tool.get("parameters", {})}}

def format_call(call: dict) -> dict:
    return {"type": "function", "function": {"name": call["name"], "arguments": call.get("arguments", {})}}

def native_row(record: dict, language: str) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPTS[language]},
        {"role": "user", "content": record["query"]},
    ]
    calls = [format_call(call) for call in record["function_calls"]]
    if calls:
        messages.append({"role": "assistant", "content": "", "tool_calls": calls})
    else:
        fallback = "Hiện tại tôi chưa thể thực hiện yêu cầu này." if language == "vi" else "I cannot complete that request right now."
        messages.append({"role": "assistant", "content": record.get("assistant_content") or fallback})
    return {"id": record["id"], "messages": messages, "tools": [format_tool(tool) for tool in record["tools"]]}

def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as s:
        return [json.loads(line) for line in s if line.strip()]

test_records = read_jsonl(REVISION_DIR / f"{split_lang}/test.jsonl")
my_records = test_records[gpu_id::total_gpus]

completed_ids = set()
if PART_PATH.exists():
    with PART_PATH.open(encoding="utf-8") as s:
        completed_ids = {json.loads(line)["id"] for line in s if line.strip()}

pending_records = [r for r in my_records if r["id"] not in completed_ids]
print(f"[{split_lang.upper()} | GPU {gpu_id}] Tổng mẫu: {len(my_records)} | Đã xong: {len(completed_ids)} | Cần chạy: {len(pending_records)}")

if pending_records:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(
        ADAPTER_ID if ADAPTER_ID else MODEL_ID,
        trust_remote_code=True,
        padding_side="left",
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"[{split_lang.upper()} | GPU {gpu_id}] Loading base model {MODEL_ID} on cuda:{gpu_id}...")
    # Qwen3.5 là kiến trúc Qwen3_5ForConditionalGeneration (image-text-to-text) có sub-module language_model.
    # Cần dùng AutoModelForImageTextToText để khớp 100% key adapter safetensors được Unsloth lưu từ Colab.
    try:
        base_model = AutoModelForImageTextToText.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map={"": gpu_id},
            dtype=torch.float16,
            trust_remote_code=True,
        )
    except Exception as e:
        print(f"[{split_lang.upper()} | GPU {gpu_id}] AutoModelForImageTextToText fallback due to: {e}")
        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map={"": gpu_id},
            dtype=torch.float16,
            trust_remote_code=True,
        )

    if ADAPTER_ID:
        print(f"[{split_lang.upper()} | GPU {gpu_id}] Loading LoRA adapter from {ADAPTER_ID}...")
        model = PeftModel.from_pretrained(
            base_model,
            ADAPTER_ID,
        ).eval()
    else:
        print(f"[{split_lang.upper()} | GPU {gpu_id}] No adapter specified, running base model...")
        model = base_model.eval()

    def prompt_text(record: dict) -> str:
        row = native_row(record, split_lang)
        return tokenizer.apply_chat_template(row["messages"][:-1], tools=row["tools"], tokenize=False, add_generation_prompt=True, enable_thinking=False)

    pending_items = [{"record": r, "prompt": prompt_text(r)} for r in pending_records]
    pending_items.sort(key=lambda x: len(x["prompt"]))

    with PART_PATH.open("a", encoding="utf-8") as output_file:
        for start in range(0, len(pending_items), BATCH_SIZE):
            batch_items = pending_items[start : start + BATCH_SIZE]
            batch_records = [item["record"] for item in batch_items]
            prompts = [item["prompt"] for item in batch_items]
            
            encoded = tokenizer(prompts, padding=True, truncation=True, max_length=4096, add_special_tokens=False, return_tensors="pt").to(f"cuda:{gpu_id}")
            t0 = time.perf_counter()
            with torch.inference_mode():
                generated = model.generate(
                    **encoded,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )
            batch_latency_ms = (time.perf_counter() - t0) * 1000.0
            generated_tokens = generated[:, encoded["input_ids"].shape[1] :]
            texts = tokenizer.batch_decode(generated_tokens, skip_special_tokens=False)

            for record, raw_output in zip(batch_records, texts, strict=True):
                output_file.write(json.dumps({
                    "id": record["id"],
                    "query": record["query"],
                    "gold": record.get("function_calls", []),
                    "raw_output": raw_output,
                    "batch_latency_ms": round(batch_latency_ms, 2),
                    "batch_size": len(batch_items),
                }, ensure_ascii=False) + "\n")
            output_file.flush()

            done = len(completed_ids) + start + len(batch_items)
            if done % 200 == 0 or done == len(my_records):
                print(f"[{split_lang.upper()} | GPU {gpu_id}] Xử lý: {done}/{len(my_records)} ({done/len(my_records)*100:.1f}%) | Latency: {batch_latency_ms:.0f}ms")

print(f"[{split_lang.upper()} | GPU {gpu_id}] ✅ HOÀN THÀNH!")

In [ ]:
# Chạy suy luận đa GPU tự động cho các ngôn ngữ trong EVAL_LANGS
for lang in EVAL_LANGS:
    print("\n" + "="*60)
    print(f"🚀 BẮT ĐẦU CHẠY SUY LUẬN TẬP: {lang.upper()} TEST")
    print("="*60)
    !python worker.py 0 2 {lang} & python worker.py 1 2 {lang} & wait

In [ ]:
import re
import json

TOOL_RE = re.compile(
    r"<tool_call>\s*<function\s*=\s*([^>\s]+)\s*>(.*?)</function>\s*</tool_call>",
    re.DOTALL | re.IGNORECASE,
)
PARAM_RE = re.compile(
    r"<parameter\s*=\s*([^>\s]+)\s*>(.*?)</parameter>",
    re.DOTALL | re.IGNORECASE,
)
JSON_CALL_RE = re.compile(
    r"<tool_call>\s*(\{.*?\})\s*</tool_call>",
    re.DOTALL | re.IGNORECASE,
)

def parse_native_output(text: str) -> tuple[list[dict], list[str]]:
    calls = []
    errors = []
    open_tags = len(re.findall(r"<tool_call\b", text, re.IGNORECASE))
    close_tags = len(re.findall(r"</tool_call\s*>", text, re.IGNORECASE))
    if open_tags != close_tags:
        errors.append("unbalanced_tool_call_tags")
        
    # 1. Thử parse theo chuẩn XML Qwen
    for name, body in TOOL_RE.findall(text):
        arguments = {}
        for parameter, value in PARAM_RE.findall(body):
            value = value.strip()
            try:
                arguments[parameter.strip()] = json.loads(value)
            except (json.JSONDecodeError, TypeError):
                arguments[parameter.strip()] = value
        calls.append({"name": name.strip(), "arguments": arguments})
        
    # 2. Fallback parse theo JSON tool call nếu model sinh dạng JSON
    if not calls:
        for json_str in JSON_CALL_RE.findall(text):
            try:
                parsed = json.loads(json_str.strip())
                if isinstance(parsed, dict) and "name" in parsed:
                    calls.append({
                        "name": parsed["name"],
                        "arguments": parsed.get("arguments", {}),
                    })
            except Exception:
                pass

    if not calls and open_tags > 0:
        errors.append("malformed_tool_call")
    return calls, errors

all_summary_metrics = {}

for lang in EVAL_LANGS:
    print("\n" + "="*60)
    print(f"📊 ĐANG GỘP VÀ CHẤM ĐIỂM CHO TẬP: {lang.upper()} TEST")
    print("="*60)
    
    # 1. Gộp các file part từ các GPU
    predictions_by_id = {}
    for gpu_id in range(torch.cuda.device_count()):
        part_path = RUN_DIR / f"eval_predictions_{lang}_part_{gpu_id}.jsonl"
        if part_path.exists():
            with part_path.open(encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        item = json.loads(line)
                        predictions_by_id[item["id"]] = item

    test_records = read_jsonl(REVISION_DIR / f"{lang}/test.jsonl")
    pred_path = RUN_DIR / f"eval_predictions_{RUN_NAME}_{lang}_test.jsonl"
    with pred_path.open("w", encoding="utf-8") as output_file:
        for record in test_records:
            rec_id = record["id"]
            if rec_id in predictions_by_id:
                output_file.write(json.dumps(predictions_by_id[rec_id], ensure_ascii=False) + "\n")
    print(f"✅ Đã gộp {len(predictions_by_id)}/{len(test_records)} mẫu vào: {pred_path.name}")

    # 2. Chấm điểm chi tiết
    rows = read_jsonl(pred_path)
    expected_ids = {r["id"] for r in test_records}
    is_complete = len(rows) == len(expected_ids)

    positive = negative = tool_correct = negative_correct = exact_match = syntax_errors = 0
    latency_ms = 0.0
    scored_rows = []
    for row in rows:
        predicted, errors = parse_native_output(row["raw_output"])
        gold = row["gold"]
        is_positive = bool(gold)
        tool_match = [call["name"] for call in predicted] == [call["name"] for call in gold]
        exact = predicted == gold
        if is_positive:
            positive += 1
            tool_correct += tool_match
        else:
            negative += 1
            negative_correct += not predicted and not errors
        exact_match += exact
        syntax_errors += bool(errors)
        latency_ms += row["batch_latency_ms"] / row["batch_size"]
        scored_rows.append({
            **row,
            "predicted": predicted,
            "errors": errors,
            "tool_match": tool_match if is_positive else (not predicted and not errors),
            "exact_match": exact,
        })

    eval_name = f"{RUN_NAME}_{lang}_test"
    metrics = {
        "split": eval_name,
        "language": lang.upper(),
        "total_samples": len(rows),
        "expected_total_samples": len(expected_ids),
        "is_complete": is_complete,
        "positive_samples": positive,
        "negative_samples": negative,
        "tool_accuracy_pos_pct": round(100 * tool_correct / positive, 2) if positive else None,
        "non_fc_recall_pct": round(100 * negative_correct / negative, 2) if negative else None,
        "arga_exact_match_pct": round(100 * exact_match / len(rows), 2) if rows else 0.0,
        "syntax_error_rate_pct": round(100 * syntax_errors / len(rows), 2) if rows else 0.0,
        "avg_latency_ms": round(latency_ms / len(rows), 2) if rows else 0.0,
    }
    all_summary_metrics[lang.upper()] = metrics

    (RUN_DIR / f"eval_predictions_{eval_name}_scored.json").write_text(
        json.dumps(scored_rows, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    (RUN_DIR / f"eval_metrics_{eval_name}.json").write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    print(json.dumps(metrics, ensure_ascii=False, indent=2))

# 3. In bảng so sánh tổng hợp đối chiếu
if len(all_summary_metrics) > 1:
    print("\n" + "="*65)
    print("🏆 BẢNG TỔNG HỢP SO SÁNH ĐỐI CHIẾU (VI vs EN BENCHMARK)")
    print("="*65)
    header = f"{'Chỉ số đánh giá (Metric)':<28} | " + " | ".join([f"{k:^14}" for k in all_summary_metrics.keys()])
    print(header)
    print("-" * len(header))
    metrics_display = [
        ("tool_accuracy_pos_pct", "Tool Selection Acc (%)"),
        ("arga_exact_match_pct", "Exact Match / ArgA (%)"),
        ("non_fc_recall_pct", "Non-FC Recall (%)"),
        ("syntax_error_rate_pct", "Syntax Error Rate (%)"),
        ("avg_latency_ms", "Avg Latency (ms)"),
    ]
    for m_key, m_label in metrics_display:
        row_str = f"{m_label:<28} | " + " | ".join([f"{str(all_summary_metrics[k].get(m_key, 'N/A')) + '%':^14}" if 'pct' in m_key else f"{str(all_summary_metrics[k].get(m_key, 'N/A')):^14}" for k in all_summary_metrics.keys()])
        print(row_str)
    print("="*65)

# ============================================================================
# 🇻🇳 PHẦN 2: ĐÁNH GIÁ TRÊN BỘ CÔNG CỤ ĐẶC THÙ VIỆT NAM (CUSTOMTOOLS-VI)
# ============================================================================
Đánh giá trên 2 tập test độc lập của CustomTools (mỗi tập 800 mẫu, gồm 400 positive + 400 negative):
1. **`test_seen.jsonl`**: Đo độ chính xác trên các domain / tools đã gặp trong quá trình train.
2. **`test_unseen.jsonl`**: Đo năng lực **Zero-shot generalization** trên các tools hoàn toàn mới chưa từng thấy.

In [ ]:
CUSTOM_DIR = Path(os.environ.get("CUSTOM_DIR", str(DATA_ROOT / "custom_vi")))

print("="*60)
print(f"Kiểm tra thư mục CustomTools: {CUSTOM_DIR}")
print("="*60)
assert CUSTOM_DIR.is_dir(), f"Thư mục không tồn tại: {CUSTOM_DIR}. Vui lòng kiểm tra lại dataset trên Kaggle."

CUSTOM_SUBSETS = ["test_seen", "test_unseen"]
for subset in CUSTOM_SUBSETS:
    path = CUSTOM_DIR / f"{subset}.jsonl"
    assert path.is_file(), f"Không tìm thấy file: {path}"
    records = read_jsonl(path)
    assert len(records) == 800, f"Số mẫu không đúng ({len(records)} != 800) tại {subset}.jsonl"
    pos = sum(1 for r in records if r.get("function_calls"))
    neg = sum(1 for r in records if not r.get("function_calls"))
    print(f"  ✓ {subset:<12}: {len(records)} mẫu (Positive: {pos}, Negative: {neg})")

print("\n✅ CustomTools-VI preflight PASS! Sẵn sàng benchmark seen & unseen.")

In [ ]:
%%writefile worker_custom.py
import os
import sys
import json
import time
from pathlib import Path
import torch
from transformers import AutoModelForImageTextToText, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

gpu_id = int(sys.argv[1])
total_gpus = int(sys.argv[2])
subset = sys.argv[3] if len(sys.argv) > 3 else "test_seen"  # "test_seen" hoặc "test_unseen"

EXPERIMENT = os.environ.get("EXPERIMENT", "e3")
MODEL_ID = os.environ.get("MODEL_ID", "unsloth/Qwen3.5-2B")
HF_NAME = os.environ.get("HF_NAME", "ThinhDao")
MODEL_NAME_ONLY = f"{MODEL_ID.rsplit('/', 1)[-1]}_{EXPERIMENT.upper()}"
ADAPTER_ID = os.environ.get("ADAPTER_ID", f"{HF_NAME}/{MODEL_NAME_ONLY}")

RUN_NAME = os.environ.get("RUN_NAME", f"{EXPERIMENT}_{MODEL_ID.rsplit('/', 1)[-1].lower()}")
RUN_DIR = Path(os.environ.get("RUN_DIR", f"/kaggle/working/{RUN_NAME}"))
RUN_DIR.mkdir(parents=True, exist_ok=True)

candidate_paths = [
    Path("/kaggle/input/datasets/phcthnho/tool-calling-vi-experiments"),
    Path("/kaggle/input/tool-calling-vi-experiments"),
]
default_data_root = next((p for p in candidate_paths if p.is_dir()), candidate_paths[0])
CUSTOM_DIR = Path(os.environ.get("CUSTOM_DIR", str(default_data_root / "custom_vi")))

PART_PATH = RUN_DIR / f"eval_predictions_{subset}_part_{gpu_id}.jsonl"

BATCH_SIZE = 16
MAX_NEW_TOKENS = 128
SYSTEM_PROMPT_VI = "Bạn là trợ lý AI có khả năng sử dụng công cụ."

def format_tool(tool: dict) -> dict:
    return {"type": "function", "function": {"name": tool["name"], "description": tool.get("description", ""), "parameters": tool.get("parameters", {})}}

def format_call(call: dict) -> dict:
    return {"type": "function", "function": {"name": call["name"], "arguments": call.get("arguments", {})}}

def native_row(record: dict) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_VI},
        {"role": "user", "content": record["query"]},
    ]
    calls = [format_call(call) for call in record.get("function_calls", [])]
    if calls:
        messages.append({"role": "assistant", "content": "", "tool_calls": calls})
    else:
        fallback = "Hiện tại tôi chưa thể thực hiện yêu cầu này."
        messages.append({"role": "assistant", "content": record.get("assistant_content") or fallback})
    return {"id": record["id"], "messages": messages, "tools": [format_tool(tool) for tool in record.get("tools", [])]}

def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as s:
        return [json.loads(line) for line in s if line.strip()]

test_records = read_jsonl(CUSTOM_DIR / f"{subset}.jsonl")
my_records = test_records[gpu_id::total_gpus]

completed_ids = set()
if PART_PATH.exists():
    with PART_PATH.open(encoding="utf-8") as s:
        completed_ids = {json.loads(line)["id"] for line in s if line.strip()}

pending_records = [r for r in my_records if r["id"] not in completed_ids]
print(f"[{subset.upper()} | GPU {gpu_id}] Tổng mẫu: {len(my_records)} | Đã xong: {len(completed_ids)} | Cần chạy: {len(pending_records)}")

if pending_records:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(
        ADAPTER_ID if ADAPTER_ID else MODEL_ID,
        trust_remote_code=True,
        padding_side="left",
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"[{subset.upper()} | GPU {gpu_id}] Loading base model {MODEL_ID} on cuda:{gpu_id}...")
    try:
        base_model = AutoModelForImageTextToText.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map={"": gpu_id},
            dtype=torch.float16,
            trust_remote_code=True,
        )
    except Exception as e:
        print(f"[{subset.upper()} | GPU {gpu_id}] AutoModelForImageTextToText fallback due to: {e}")
        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map={"": gpu_id},
            dtype=torch.float16,
            trust_remote_code=True,
        )

    if ADAPTER_ID:
        print(f"[{subset.upper()} | GPU {gpu_id}] Loading LoRA adapter from {ADAPTER_ID}...")
        model = PeftModel.from_pretrained(base_model, ADAPTER_ID).eval()
    else:
        print(f"[{subset.upper()} | GPU {gpu_id}] No adapter specified, running base model...")
        model = base_model.eval()

    def prompt_text(record: dict) -> str:
        row = native_row(record)
        return tokenizer.apply_chat_template(row["messages"][:-1], tools=row["tools"], tokenize=False, add_generation_prompt=True, enable_thinking=False)

    pending_items = [{"record": r, "prompt": prompt_text(r)} for r in pending_records]
    pending_items.sort(key=lambda x: len(x["prompt"]))

    with PART_PATH.open("a", encoding="utf-8") as output_file:
        for start in range(0, len(pending_items), BATCH_SIZE):
            batch_items = pending_items[start : start + BATCH_SIZE]
            batch_records = [item["record"] for item in batch_items]
            prompts = [item["prompt"] for item in batch_items]

            encoded = tokenizer(prompts, padding=True, truncation=True, max_length=4096, add_special_tokens=False, return_tensors="pt").to(f"cuda:{gpu_id}")
            t0 = time.perf_counter()
            with torch.inference_mode():
                generated = model.generate(
                    **encoded,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )
            batch_latency_ms = (time.perf_counter() - t0) * 1000.0
            generated_tokens = generated[:, encoded["input_ids"].shape[1] :]
            texts = tokenizer.batch_decode(generated_tokens, skip_special_tokens=False)

            for record, raw_output in zip(batch_records, texts, strict=True):
                output_file.write(json.dumps({
                    "id": record["id"],
                    "query": record["query"],
                    "gold": record.get("function_calls", []),
                    "raw_output": raw_output,
                    "batch_latency_ms": round(batch_latency_ms, 2),
                    "batch_size": len(batch_items),
                }, ensure_ascii=False) + "\n")
            output_file.flush()

            done = len(completed_ids) + start + len(batch_items)
            if done % 100 == 0 or done == len(my_records):
                print(f"[{subset.upper()} | GPU {gpu_id}] Xử lý: {done}/{len(my_records)} ({done/len(my_records)*100:.1f}%) | Latency: {batch_latency_ms:.0f}ms")

print(f"[{subset.upper()} | GPU {gpu_id}] ✅ HOÀN THÀNH!")


In [ ]:
# Chạy suy luận đa GPU cho cả 2 tập test của CustomTools (Seen & Unseen)
for subset in CUSTOM_SUBSETS:
    print("\n" + "="*60)
    print(f"🚀 BẮT ĐẦU CHẠY SUY LUẬN CUSTOM: {subset.upper()}")
    print("="*60)
    !python worker_custom.py 0 2 {subset} & python worker_custom.py 1 2 {subset} & wait

In [ ]:
custom_summary_metrics = {}

for subset in CUSTOM_SUBSETS:
    print("\n" + "="*60)
    print(f"📊 ĐANG GỘP VÀ CHẤM ĐIỂM CHO CUSTOM: {subset.upper()}")
    print("="*60)

    # 1. Gộp các file part từ các GPU
    predictions_by_id = {}
    for gpu_id in range(torch.cuda.device_count()):
        part_path = RUN_DIR / f"eval_predictions_{subset}_part_{gpu_id}.jsonl"
        if part_path.exists():
            with part_path.open(encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        item = json.loads(line)
                        predictions_by_id[item["id"]] = item

    test_records = read_jsonl(CUSTOM_DIR / f"{subset}.jsonl")
    pred_path = RUN_DIR / f"eval_predictions_{RUN_NAME}_{subset}.jsonl"
    with pred_path.open("w", encoding="utf-8") as output_file:
        for record in test_records:
            rec_id = record["id"]
            if rec_id in predictions_by_id:
                output_file.write(json.dumps(predictions_by_id[rec_id], ensure_ascii=False) + "\n")
    print(f"✅ Đã gộp {len(predictions_by_id)}/{len(test_records)} mẫu vào: {pred_path.name}")

    # 2. Chấm điểm chi tiết
    rows = read_jsonl(pred_path)
    expected_ids = {r["id"] for r in test_records}
    is_complete = len(rows) == len(expected_ids)

    positive = negative = tool_correct = negative_correct = exact_match = syntax_errors = 0
    latency_ms = 0.0
    scored_rows = []
    for row in rows:
        predicted, errors = parse_native_output(row["raw_output"])
        gold = row["gold"]
        is_positive = bool(gold)
        tool_match = [call["name"] for call in predicted] == [call["name"] for call in gold]
        exact = predicted == gold
        if is_positive:
            positive += 1
            tool_correct += tool_match
        else:
            negative += 1
            negative_correct += not predicted and not errors
        exact_match += exact
        syntax_errors += bool(errors)
        latency_ms += row["batch_latency_ms"] / row["batch_size"]
        scored_rows.append({
            **row,
            "predicted": predicted,
            "errors": errors,
            "tool_match": tool_match if is_positive else (not predicted and not errors),
            "exact_match": exact,
        })

    eval_name = f"{RUN_NAME}_{subset}"
    metrics = {
        "split": eval_name,
        "subset": subset,
        "total_samples": len(rows),
        "expected_total_samples": len(expected_ids),
        "is_complete": is_complete,
        "positive_samples": positive,
        "negative_samples": negative,
        "tool_accuracy_pos_pct": round(100 * tool_correct / positive, 2) if positive else None,
        "non_fc_recall_pct": round(100 * negative_correct / negative, 2) if negative else None,
        "arga_exact_match_pct": round(100 * exact_match / len(rows), 2) if rows else 0.0,
        "syntax_error_rate_pct": round(100 * syntax_errors / len(rows), 2) if rows else 0.0,
        "avg_latency_ms": round(latency_ms / len(rows), 2) if rows else 0.0,
    }
    custom_summary_metrics[subset] = metrics

    (RUN_DIR / f"eval_predictions_{eval_name}_scored.json").write_text(
        json.dumps(scored_rows, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    (RUN_DIR / f"eval_metrics_{eval_name}.json").write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    print(json.dumps(metrics, ensure_ascii=False, indent=2))

# 3. In bảng so sánh đối chiếu Seen vs Unseen
if len(custom_summary_metrics) > 1:
    print("\n" + "="*65)
    print("🏆 BẢNG TỔNG HỢP SO SÁNH CUSTOMTOOLS-VI (SEEN vs UNSEEN)")
    print("="*65)
    header = f"{'Chỉ số đánh giá (Metric)':<28} | " + " | ".join([f"{k:^16}" for k in custom_summary_metrics.keys()])
    print(header)
    print("-" * len(header))
    metrics_display = [
        ("tool_accuracy_pos_pct", "Tool Selection Acc (%)"),
        ("arga_exact_match_pct", "Exact Match / ArgA (%)"),
        ("non_fc_recall_pct", "Non-FC Recall (%)"),
        ("syntax_error_rate_pct", "Syntax Error Rate (%)"),
        ("avg_latency_ms", "Avg Latency (ms)"),
    ]
    for m_key, m_label in metrics_display:
        row_str = f"{m_label:<28} | " + " | ".join([f"{str(custom_summary_metrics[k].get(m_key, 'N/A')) + '%':^16}" if 'pct' in m_key else f"{str(custom_summary_metrics[k].get(m_key, 'N/A')):^16}" for k in custom_summary_metrics.keys()])
        print(row_str)
    print("="*65)
